# 🌿 AgriVision AI: ConvNeXt V2 Full Training on ALL PlantSeg Classes (Kaggle P100 GPU)
### End-to-End Multimodal Plant Disease Model
- **Dataset:** `weitianqi/plantseg` (All 11,400+ in-the-wild images, all 115 diseases)
- **Hardware:** Kaggle NVIDIA Tesla P100 GPU (16 GB VRAM)
- **Speed Optimization:** PyTorch Mixed Precision (AMP FP16) for 2x faster training (~20-25 mins total)
- **Outputs:** `best_convnextv2_all_classes.pth`, `disease_classes.json`, Grad-CAM++ visualizations.

In [ ]:
# Step 1: Verify Kaggle GPU
!nvidia-smi

In [ ]:
# Step 2: Install required vision libraries (Make sure Internet is ON in Kaggle settings!)
!pip install -q timm grad-cam

In [ ]:
# Step 3: Locate Dataset on Kaggle
import os
import pandas as pd

kaggle_input_dir = "/kaggle/input"
csv_path = None

for root, dirs, files in os.walk(kaggle_input_dir):
    for f in files:
        if f.lower().endswith('.csv'):
            csv_path = os.path.join(root, f)
            break
    if csv_path:
        break

print(f"Found Metadata at: {csv_path}")
df = pd.read_csv(csv_path)
print(f"Total dataset records: {len(df)}")
print(f"Total unique diseases: {df['Disease'].nunique()}")
print("Sample diseases:", df['Disease'].unique()[:5].tolist())

In [ ]:
# Step 4: Index All 11,400+ Images and Build DataLoader
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Index all image file paths on disk
print("Indexing image files across Kaggle input...")
file_map = {}
for root, dirs, files in os.walk(kaggle_input_dir):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            file_map[f] = os.path.join(root, f)

print(f"Indexed {len(file_map)} total image files!")

# Get all unique disease classes sorted alphabetically for consistent mapping
all_classes = sorted(df['Disease'].dropna().unique().tolist())
label_to_idx = {name: idx for idx, name in enumerate(all_classes)}

# Save class names list to JSON for the Streamlit app
with open('/kaggle/working/disease_classes.json', 'w') as f:
    json.dump(all_classes, f, indent=2)
print(f"Saved {len(all_classes)} class mappings to disease_classes.json")

class FullPlantSegDataset(Dataset):
    def __init__(self, df, file_map, label_to_idx, transform=None):
        self.samples = []
        self.transform = transform
        for _, row in df.iterrows():
            img_name = str(row['Name'])
            dis_name = str(row['Disease'])
            if img_name in file_map and dis_name in label_to_idx:
                self.samples.append((file_map[img_name], label_to_idx[dis_name]))
        self.classes = all_classes

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), color=0)
        if self.transform:
            img = self.transform(img)
        return img, label

# High performance augmentations & normalizations
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = FullPlantSegDataset(df, file_map, label_to_idx, transform=train_tf)
print(f"\n Loaded {len(dataset)} valid images across {len(all_classes)} disease categories!")

# 85% Train / 15% Validation split
train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
print(f"Batches: {len(train_loader)} Train batches, {len(val_loader)} Val batches (Batch Size: 64)")

In [ ]:
# Step 5: Instantiate ConvNeXt V2 Architecture
import timm
import torch.nn as nn

model_name = 'convnextv2_tiny'
model = timm.create_model(model_name, pretrained=True, num_classes=len(all_classes))
model = model.to(device)
print(f"Initialized {model_name} with {len(all_classes)} output classification heads!")

In [ ]:
# Step 6: Accelerated Training with PyTorch Mixed Precision (AMP FP16)
from torch.cuda.amp import GradScaler, autocast

criterion = nn.CrossEntropyLoss(label_smoothing=0.1) # Label smoothing handles small classes smoothly
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
epochs = 8
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = GradScaler()

best_acc = 0.0
save_path = '/kaggle/working/best_convnextv2_all_classes.pth'

print(f"=== Starting High-Speed FP16 Training on Kaggle P100 ({epochs} Epochs) ===\n")

for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        # Mixed precision forward
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()
        total += labels.size(0)
        
    scheduler.step()
    train_loss = running_loss / total
    train_acc = correct / total
    
    # Validation Loop
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for vimages, vlabels in val_loader:
            vimages, vlabels = vimages.to(device), vlabels.to(device)
            with autocast():
                voutputs = model(vimages)
            _, vpreds = torch.max(voutputs, 1)
            val_correct += torch.sum(vpreds == vlabels.data).item()
            val_total += vlabels.size(0)
            
    val_acc = val_correct / val_total if val_total > 0 else 0.0
    print(f"Epoch [{epoch:02d}/{epochs:02d}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"   --> Checkpoint saved (New Best: {best_acc*100:.2f}%)")

print(f"\n Training Completed! Best Validation Accuracy across all classes: {best_acc*100:.2f}%")

In [ ]:
# Step 7: Explainable AI with Grad-CAM++ on Kaggle
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np
import matplotlib.pyplot as plt

target_layers = [model.stages[-1].blocks[-1]]
cam = GradCAMPlusPlus(model=model, target_layers=target_layers)

sample_img, sample_lbl = val_ds[0]
input_tensor = sample_img.unsqueeze(0).to(device)
targets = [ClassifierOutputTarget(sample_lbl)]

grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]
inv_norm_img = sample_img.permute(1, 2, 0).cpu().numpy()
inv_norm_img = (inv_norm_img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
inv_norm_img = np.clip(inv_norm_img, 0, 1)

visualization = show_cam_on_image(inv_norm_img, grayscale_cam, use_rgb=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(inv_norm_img)
axes[0].set_title(f"Kaggle Leaf: {all_classes[sample_lbl]}")
axes[0].axis('off')

axes[1].imshow(visualization)
axes[1].set_title("Grad-CAM++ Explainability Heatmap")
axes[1].axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/gradcam_all_classes.png')
plt.show()
print("Grad-CAM++ Heatmap generated successfully!")

In [ ]:
# Step 8: View Output Files Ready for Download
print("Your output files in /kaggle/working/:")
for f in os.listdir('/kaggle/working'):
    sz = os.path.getsize(os.path.join('/kaggle/working', f)) / (1024 * 1024)
    print(f"  - {f} ({sz:.2f} MB)")
print("\nYou can now download 'best_convnextv2_all_classes.pth' and 'disease_classes.json' from the Kaggle Output panel on the right!")